# 04 - Feature Engineering (PRO ⚡)

In [17]:

import pandas as pd
import numpy as np
from pathlib import Path


In [18]:

base_path = Path(r'C:/Users/rroman/OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)/Documents/Unidad/2026/Capacitación/Data Science y Machine Learning/proyecto final/data')
input_path = base_path / 'processed' / 'dataset_pivot.parquet'
output_path = base_path / 'processed' / 'dataset_features.parquet'

df = pd.read_parquet(input_path)

print("Shape inicial:", df.shape)
df.head()


Shape inicial: (10258370, 15)


,Meter ID,Datetime,CurrentPhaseA,CurrentPhaseB,CurrentPhaseC,CurrentPhaseN,VoltagePhaseA,VoltagePhaseB,VoltagePhaseC,VoltageSag,VoltageSwell,kVARh_Del,kVARh_Rec,kWh_Del,kWh_Rec
0,F-74749,2026-02-09 00:15:00,0.0923,0.0924,0.0846,0.0,114.7291,113.1553,111.8357,0.0,0.0,0.0035,0.0,0.0071,0.0
1,F-74749,2026-02-09 00:30:00,0.0960,0.1010,0.0849,0.0,114.4624,113.5427,111.6652,0.0,0.0,0.0039,0.0,0.0073,0.0
2,F-74749,2026-02-09 00:45:00,0.0842,0.0880,0.0746,0.0,114.3094,113.4678,111.4966,0.0,0.0,0.0034,0.0,0.0065,0.0
3,F-74749,2026-02-09 01:00:00,0.0983,0.1055,0.0884,0.0,114.3358,113.4620,111.5659,0.0,0.0,0.0040,0.0,0.0075,0.0
4,F-74749,2026-02-09 01:15:00,0.0883,0.0867,0.0818,0.0,114.3860,113.5381,111.5726,0.0,0.0,0.0033,0.0,0.0068,0.0


In [19]:

# Voltaje
df["Voltage_avg"] = df[["VoltagePhaseA","VoltagePhaseB","VoltagePhaseC"]].mean(axis=1)
df["Voltage_unbalance"] = df[["VoltagePhaseA","VoltagePhaseB","VoltagePhaseC"]].std(axis=1)
df["Voltage_min"] = df[["VoltagePhaseA","VoltagePhaseB","VoltagePhaseC"]].min(axis=1)
df["Voltage_max"] = df[["VoltagePhaseA","VoltagePhaseB","VoltagePhaseC"]].max(axis=1)


In [21]:

# Corriente
df["Current_avg"] = df[["CurrentPhaseA","CurrentPhaseB","CurrentPhaseC"]].mean(axis=1)
df["Current_unbalance"] = df[["CurrentPhaseA","CurrentPhaseB","CurrentPhaseC"]].std(axis=1)


In [22]:

# Energía → Potencia
df["Power_kW"] = df["kWh_Del"] * 4
df["ReactivePower_kVAR"] = df["kVARh_Del"] * 4


In [23]:

# Factor de potencia
df["PF"] = df["kWh_Del"] / np.sqrt(df["kWh_Del"]**2 + df["kVARh_Del"]**2)


In [24]:

# SAG / SWELL
df["Sag_duration_sec"] = df["VoltageSag"] * 0.3
df["Swell_duration_sec"] = df["VoltageSwell"] * 0.3

df["Sag_pct"] = df["Sag_duration_sec"] / (15*60)
df["Swell_pct"] = df["Swell_duration_sec"] / (15*60)

df["Disturbance_score"] = df["Sag_pct"] + df["Swell_pct"]


In [25]:

# Variables temporales
df["Datetime"] = pd.to_datetime(df["Datetime"])
df["hour"] = df["Datetime"].dt.hour
df["dayofweek"] = df["Datetime"].dt.dayofweek


In [26]:

# Dinámica
df = df.sort_values(["Meter ID","Datetime"])

df["Voltage_drop"] = df.groupby("Meter ID")["Voltage_avg"].diff()
df["Current_spike"] = df.groupby("Meter ID")["Current_avg"].diff()


In [27]:

# Evento de disturbio
df["Voltage_event"] = ((df["VoltageSag"] > 0) | (df["VoltageSwell"] > 0)).astype(int)


In [28]:

# Guardar
df.to_parquet(output_path, index=False)

print("Dataset generado")
df.head()


Dataset generado


,Meter ID,Datetime,CurrentPhaseA,CurrentPhaseB,CurrentPhaseC,CurrentPhaseN,VoltagePhaseA,VoltagePhaseB,VoltagePhaseC,VoltageSag,...,Sag_duration_sec,Swell_duration_sec,Sag_pct,Swell_pct,Disturbance_score,hour,dayofweek,Voltage_drop,Current_spike,Voltage_event
0,F-74749,2026-02-09 00:15:00,0.0923,0.0924,0.0846,0.0,114.7291,113.1553,111.8357,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,NaN,NaN,0
1,F-74749,2026-02-09 00:30:00,0.0960,0.1010,0.0849,0.0,114.4624,113.5427,111.6652,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,-0.016600,0.004200,0
2,F-74749,2026-02-09 00:45:00,0.0842,0.0880,0.0746,0.0,114.3094,113.4678,111.4966,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0,-0.132167,-0.011700,0
3,F-74749,2026-02-09 01:00:00,0.0983,0.1055,0.0884,0.0,114.3358,113.4620,111.5659,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,0.029967,0.015133,0
4,F-74749,2026-02-09 01:15:00,0.0883,0.0867,0.0818,0.0,114.3860,113.5381,111.5726,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,0.044333,-0.011800,0
